# Stage 20 V1 — Ролевой интерпретатор результата GPT

## Вопрос

Может ли GPT в строго ограниченном режиме интерпретировать уже рассчитанный синтетический результат для четырёх получателей, не изменяя score, threshold или кредитное решение?

Это controlled prototype: две frozen synthetic cards × четыре роли = ровно восемь независимых вызовов OpenAI Responses API. GPT — только интерпретатор результата; он не рассчитывает риск, не принимает кредитное решение, не использует реальные клиентские данные и не получает внешние инструменты.

In [2]:
# 1.1 — Импорты и детерминированные утилиты
import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from pathlib import Path

from openai import OpenAI
import openai

def find_project_root(start_path):
    for candidate in (start_path.resolve(), *start_path.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'reports').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('Не удалось определить корень проекта: не найдена директория с pyproject.toml, reports/ и notebooks/.')

ROOT = find_project_root(Path.cwd())
ARTIFACT_PATH = ROOT / 'reports' / 'generated' / 'stage20_role_interpreter_V1.json'
MODEL_ID = 'gpt-5.6-luna'
ROLES = ('sales_manager', 'credit_controller', 'lawyer', 'information_security')

def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False)

def sha256_text(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()

def utc_now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')


## ФАКТЫ

Карточки ниже заданы локально до обращения к API. Для одной карточки между запросами меняется только `recipient_role`; остальные поля сериализуются в canonical JSON.

## ИНТЕРПРЕТАЦИЯ

Ролевая адаптация относится к способу объяснения, а не к расчёту или изменению модельного результата.

## ОГРАНИЧЕНИЯ

Синтетические карточки не являются реальными клиентскими данными. Этот notebook не доказывает качество кредитного решения, юридическую достаточность, security readiness или production readiness.

## СЛЕДУЮЩИЙ ШАГ

Зафиксировать входной контракт, output schema и provenance-проверки до API-вызовов.

In [3]:
# 2.1 — Frozen cards, master instruction и строгие JSON Schema

INPUT_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'required': [
        'scenario_id',
        'synthetic',
        'model_result',
        'local_factors',
        'limitations',
        'execution_context',
        'recipient_role',
    ],
    'properties': {
        'scenario_id': {
            'type': 'string',
            'enum': ['CASE_A', 'CASE_B'],
        },
        'synthetic': {
            'type': 'boolean',
        },
        'model_result': {
            'type': 'object',
            'additionalProperties': False,
            'required': [
                'model_name',
                'score',
                'score_semantics',
                'decision_mode',
                'threshold',
                'relative_to_threshold',
                'ml_decision',
                'additional_review',
            ],
            'properties': {
                'model_name': {'type': 'string'},
                'score': {'type': 'number'},
                'score_semantics': {'type': 'string'},
                'decision_mode': {'type': 'string'},
                'threshold': {'type': 'number'},
                'relative_to_threshold': {
                    'type': 'string',
                    'enum': ['below', 'equal', 'above'],
                },
                'ml_decision': {'type': 'string'},
                'additional_review': {'type': 'boolean'},
            },
        },
        'local_factors': {
            'type': 'array',
            'minItems': 2,
            'maxItems': 2,
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'required': [
                    'factor_id',
                    'label',
                    'direction',
                    'provided_explanation',
                ],
                'properties': {
                    'factor_id': {'type': 'string'},
                    'label': {'type': 'string'},
                    'direction': {
                        'type': 'string',
                        'enum': [
                            'raises_score',
                            'lowers_score',
                            'neutral',
                        ],
                    },
                    'provided_explanation': {'type': 'string'},
                },
            },
        },
        'limitations': {
            'type': 'array',
            'minItems': 1,
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'required': [
                    'limitation_id',
                    'text',
                ],
                'properties': {
                    'limitation_id': {'type': 'string'},
                    'text': {'type': 'string'},
                },
            },
        },
        'execution_context': {
            'type': 'object',
            'additionalProperties': False,
            'required': [
                'real_client_data_used',
                'external_tools_provided',
                'web_search_provided',
                'file_search_provided',
                'store',
            ],
            'properties': {
                'real_client_data_used': {'type': 'boolean'},
                'external_tools_provided': {'type': 'boolean'},
                'web_search_provided': {'type': 'boolean'},
                'file_search_provided': {'type': 'boolean'},
                'store': {'type': 'boolean'},
            },
        },
        'recipient_role': {
            'type': 'string',
            'enum': list(ROLES),
        },
    },
}


# Полные допустимые идентификаторы источников.
# GPT обязан возвращать именно эти строки, без сокращений.
SOURCE_FACT_ID_ENUM = [
    'model_result.model_name',
    'model_result.score',
    'model_result.score_semantics',
    'model_result.decision_mode',
    'model_result.threshold',
    'model_result.relative_to_threshold',
    'model_result.ml_decision',
    'model_result.additional_review',

    'execution_context.real_client_data_used',
    'execution_context.external_tools_provided',
    'execution_context.web_search_provided',
    'execution_context.file_search_provided',
    'execution_context.store',

    'local_factors.LF_A_PAYMENT_DISCIPLINE',
    'local_factors.LF_A_FINANCIAL_STABILITY',
    'local_factors.LF_B_PAYMENT_DISCIPLINE',
    'local_factors.LF_B_FINANCIAL_STABILITY',

    'limitations.LIM_SYNTHETIC_ONLY',
    'limitations.LIM_NO_CREDIT_DECISION',
    'limitations.LIM_NO_RISK_CALCULATION',
    'limitations.LIM_NO_EXTERNAL_TOOLS',

    'synthetic',
    'scenario_id',
]


FACTOR_ID_ENUM = [
    'LF_A_PAYMENT_DISCIPLINE',
    'LF_A_FINANCIAL_STABILITY',
    'LF_B_PAYMENT_DISCIPLINE',
    'LF_B_FINANCIAL_STABILITY',
]


OUTPUT_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'required': [
        'scenario_id',
        'recipient_role',
        'preserved_result',
        'statements',
        'used_factor_ids',
    ],
    'properties': {
        'scenario_id': {
            'type': 'string',
            'enum': ['CASE_A', 'CASE_B'],
        },
        'recipient_role': {
            'type': 'string',
            'enum': list(ROLES),
        },
        'preserved_result': {
            'type': 'object',
            'additionalProperties': False,
            'required': [
                'model_name',
                'score',
                'threshold',
                'relative_to_threshold',
                'ml_decision',
                'additional_review',
            ],
            'properties': {
                'model_name': {'type': 'string'},
                'score': {'type': 'number'},
                'threshold': {'type': 'number'},
                'relative_to_threshold': {
                    'type': 'string',
                    'enum': ['below', 'equal', 'above'],
                },
                'ml_decision': {'type': 'string'},
                'additional_review': {'type': 'boolean'},
            },
        },
        'statements': {
            'type': 'array',
            'minItems': 1,
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'required': [
                    'kind',
                    'text',
                    'source_fact_ids',
                ],
                'properties': {
                    'kind': {
                        'type': 'string',
                        'enum': [
                            'fact',
                            'workflow_interpretation',
                            'limitation',
                        ],
                    },
                    'text': {
                        'type': 'string',
                    },
                    'source_fact_ids': {
                        'type': 'array',
                        'minItems': 1,
                        'items': {
                            'type': 'string',
                            'enum': SOURCE_FACT_ID_ENUM,
                        },
                    },
                },
            },
        },
        'used_factor_ids': {
            'type': 'array',
            'items': {
                'type': 'string',
                'enum': FACTOR_ID_ENUM,
            },
        },
    },
}


MASTER_INSTRUCTION = '''Вы — Result Interpreter в контролируемом прототипе. Пишите по-русски.

Ваша задача — только интерпретировать уже рассчитанный результат модели для указанной роли.

Вы НЕ:
- рассчитываете кредитный риск;
- не меняете score;
- не меняете threshold;
- не меняете ml_decision;
- не принимаете кредитное или юридическое решение;
- не добавляете отсутствующие факты;
- не используете внешние инструменты или источники.

Верните только JSON, соответствующий переданной JSON Schema.

preserved_result должен буквально сохранять значения из model_result входной карточки.

Каждый объект в statements обязан иметь непустой source_fact_ids.

ВАЖНО:
source_fact_ids — это технические canonical ID.
Используйте только ПОЛНЫЕ идентификаторы из JSON Schema.

Правильно:
- model_result.score
- model_result.threshold
- model_result.ml_decision
- model_result.relative_to_threshold
- model_result.additional_review
- local_factors.LF_A_PAYMENT_DISCIPLINE
- limitations.LIM_SYNTHETIC_ONLY

Неправильно:
- score
- threshold
- ml_decision
- SYNTHETIC_GBDT_DEMO
- LF_A_PAYMENT_DISCIPLINE
- LIM_SYNTHETIC_ONLY

Никогда не сокращайте canonical ID.

Для local_factors используйте только factor_id, реально присутствующие в текущей входной карточке.
Для limitations используйте только limitation_id, реально присутствующие в текущей входной карточке.

Не вводите в текст новых числовых значений, которых нет во входной карточке.

Ролевые правила:

sales_manager:
дайте короткое и понятное объяснение результата и его значения для указанного workflow.
Не перегружайте техническими деталями.

credit_controller:
отразите score, threshold, положение относительно threshold, additional_review,
переданные local factors и ограничения.

lawyer:
используйте только подтверждённые входом факты.
Отразите происхождение результата и ограничения.
Не делайте причинных утверждений и не формулируйте самостоятельное юридическое решение.

information_security:
укажите только фактически переданный execution_context,
синтетический характер карточки и отсутствие предоставленных external tools.

store=false означает только значение параметра запроса.
Не утверждайте, что это означает:
- отсутствие хранения данных вообще;
- Zero Data Retention;
- гарантированное отсутствие retention.

Используйте только сведения из текущего input.'''


COMMON_LIMITATIONS = [
    {
        'limitation_id': 'LIM_SYNTHETIC_ONLY',
        'text': 'Карточка синтетическая и не содержит реальных клиентских данных.',
    },
    {
        'limitation_id': 'LIM_NO_CREDIT_DECISION',
        'text': 'Интерпретация не является кредитным решением.',
    },
    {
        'limitation_id': 'LIM_NO_RISK_CALCULATION',
        'text': 'GPT не рассчитывает кредитный риск.',
    },
    {
        'limitation_id': 'LIM_NO_EXTERNAL_TOOLS',
        'text': 'Внешние инструменты не предоставлены.',
    },
]


BASE_CONTEXT = {
    'real_client_data_used': False,
    'external_tools_provided': False,
    'web_search_provided': False,
    'file_search_provided': False,
    'store': False,
}


BASE_MODEL = {
    'model_name': 'SYNTHETIC_GBDT_DEMO',
    'score_semantics': 'Синтетический модельный score из frozen card.',
    'decision_mode': 'Зафиксированный workflow-режим из frozen card.',
}


FROZEN_CARDS = {
    'CASE_A': {
        'scenario_id': 'CASE_A',
        'synthetic': True,
        'model_result': {
            **BASE_MODEL,
            'score': 0.32,
            'threshold': 0.50,
            'relative_to_threshold': 'below',
            'ml_decision': 'standard_processing',
            'additional_review': False,
        },
        'local_factors': [
            {
                'factor_id': 'LF_A_PAYMENT_DISCIPLINE',
                'label': 'Платёжная дисциплина',
                'direction': 'neutral',
                'provided_explanation': (
                    'Фактор передан локально: стабильная платёжная дисциплина.'
                ),
            },
            {
                'factor_id': 'LF_A_FINANCIAL_STABILITY',
                'label': 'Финансовая устойчивость',
                'direction': 'neutral',
                'provided_explanation': (
                    'Фактор передан локально: финансовая устойчивость без отмеченных отклонений.'
                ),
            },
        ],
        'limitations': COMMON_LIMITATIONS,
        'execution_context': BASE_CONTEXT,
    },

    'CASE_B': {
        'scenario_id': 'CASE_B',
        'synthetic': True,
        'model_result': {
            **BASE_MODEL,
            'score': 0.49,
            'threshold': 0.50,
            'relative_to_threshold': 'below',
            'ml_decision': 'additional_review',
            'additional_review': True,
        },
        'local_factors': [
            {
                'factor_id': 'LF_B_PAYMENT_DISCIPLINE',
                'label': 'Платёжная дисциплина',
                'direction': 'neutral',
                'provided_explanation': (
                    'Фактор передан локально: платёжная дисциплина требует дополнительного внимания.'
                ),
            },
            {
                'factor_id': 'LF_B_FINANCIAL_STABILITY',
                'label': 'Финансовая устойчивость',
                'direction': 'neutral',
                'provided_explanation': (
                    'Фактор передан локально: финансовая устойчивость требует дополнительного внимания.'
                ),
            },
        ],
        'limitations': COMMON_LIMITATIONS,
        'execution_context': BASE_CONTEXT,
    },
}


# Локальные проверки frozen contract.
assert set(FROZEN_CARDS) == {'CASE_A', 'CASE_B'}

assert all(
    len(card['local_factors']) == 2
    for card in FROZEN_CARDS.values()
)

assert all(
    card['synthetic']
    and not card['execution_context']['real_client_data_used']
    for card in FROZEN_CARDS.values()
)

assert (
    OUTPUT_SCHEMA['properties']['statements']
    ['items']['properties']['source_fact_ids']
    ['items']['enum']
    == SOURCE_FACT_ID_ENUM
)

MASTER_PROMPT_SHA256 = sha256_text(MASTER_INSTRUCTION)


def card_for_role(card, role):
    return {
        **card,
        'recipient_role': role,
    }


canonical_inputs = {
    (scenario_id, role): canonical_json(
        card_for_role(card, role)
    )
    for scenario_id, card in FROZEN_CARDS.items()
    for role in ROLES
}


for scenario_id in FROZEN_CARDS:
    model_hashes = {
        sha256_text(
            canonical_json(
                card_for_role(
                    FROZEN_CARDS[scenario_id],
                    role,
                )['model_result']
            )
        )
        for role in ROLES
    }

    assert len(model_hashes) == 1


print(
    f'Frozen cards: {len(FROZEN_CARDS)}; '
    f'planned independent requests: {len(canonical_inputs)}'
)

Frozen cards: 2; planned independent requests: 8


## ФАКТЫ

`CASE_A` имеет score 0.32 и workflow `standard_processing`; `CASE_B` — score 0.49 и workflow `additional_review`. Оба score находятся ниже threshold 0.50 по уже заданному полю `relative_to_threshold`.

## ИНТЕРПРЕТАЦИЯ

Близость score к threshold не рассчитывается GPT: для `CASE_B` workflow и review flag уже переданы в карточке.

## ОГРАНИЧЕНИЯ

Запрещённые источники provenance, новые factor ID и новые числовые значения должны быть отклонены автоматической проверкой.

## СЛЕДУЮЩИЙ ШАГ

Определить детерминированные проверки для каждого из восьми ответов.

In [4]:
# 3.1 — Детерминированная автоматическая валидация
BASE_SOURCE_FACT_IDS = {
    'model_result.model_name', 'model_result.score', 'model_result.score_semantics', 'model_result.decision_mode',
    'model_result.threshold', 'model_result.relative_to_threshold', 'model_result.ml_decision', 'model_result.additional_review',
    'execution_context.real_client_data_used', 'execution_context.external_tools_provided',
    'execution_context.web_search_provided', 'execution_context.file_search_provided', 'execution_context.store',
    'synthetic', 'scenario_id'
}
NUMBER_RE = re.compile(r'(?<![A-Za-zА-Яа-я_])[-+]?(?:\d+[.,]\d+|\d+)(?![A-Za-zА-Яа-я_])')

def numeric_whitelist(card):
    return {Decimal(str(card['model_result']['score'])), Decimal(str(card['model_result']['threshold']))}

def normalized_numbers(text):
    values = set()
    for token in NUMBER_RE.findall(text):
        try:
            values.add(Decimal(token.replace(',', '.')))
        except InvalidOperation:
            return None
    return values

def numbers_are_allowed(card, text):
    values = normalized_numbers(text)
    return values is not None and values.issubset(numeric_whitelist(card))

def response_has_tool_call(response):
    return any(getattr(item, 'type', '') in {'function_call', 'computer_call', 'file_search_call', 'web_search_call', 'mcp_call'} for item in (response.output or []))

def validate_response(card, role, response, parsed):
    expected = card_for_role(card, role)
    factor_ids = {factor['factor_id'] for factor in card['local_factors']}
    limitation_ids = {limitation['limitation_id'] for limitation in card['limitations']}
    allowed_source_ids = BASE_SOURCE_FACT_IDS | {f'local_factors.{factor_id}' for factor_id in factor_ids} | {f'limitations.{limitation_id}' for limitation_id in limitation_ids}
    errors = []
    required = set(OUTPUT_SCHEMA['required'])
    if set(parsed) != required:
        errors.append('Нарушен верхний уровень output schema.')
    if parsed.get('scenario_id') != expected['scenario_id']:
        errors.append('scenario_id не совпадает с input.')
    if parsed.get('recipient_role') != role:
        errors.append('recipient_role не совпадает с input.')
    preserved_expected = {key: expected['model_result'][key] for key in ('model_name', 'score', 'threshold', 'relative_to_threshold', 'ml_decision', 'additional_review')}
    if parsed.get('preserved_result') != preserved_expected:
        errors.append('preserved_result не равен model_result из input.')
    statements = parsed.get('statements')
    if not isinstance(statements, list):
        errors.append('statements не является list.')
    else:
        for statement_index, statement in enumerate(statements):
            source_fact_ids = statement.get('source_fact_ids') if isinstance(statement, dict) else None
            if not isinstance(source_fact_ids, list) or len(source_fact_ids) < 1:
                errors.append(f'statement[{statement_index}] имеет пустой или некорректный source_fact_ids.')
            elif any(source_id not in allowed_source_ids for source_id in source_fact_ids):
                errors.append(f'statement[{statement_index}] содержит source_fact_id вне whitelist.')
    used_factor_ids = parsed.get('used_factor_ids', [])
    if not isinstance(used_factor_ids, list) or any(factor_id not in factor_ids for factor_id in used_factor_ids):
        errors.append('used_factor_ids выходят за local_factors input.')
    mentioned_factor_ids = set(re.findall(r'LF_[A-Z_]+', canonical_json(parsed)))
    if not mentioned_factor_ids.issubset(factor_ids):
        errors.append('В ответе упомянут несуществующий factor ID.')
    statement_text = ' '.join(statement.get('text', '') for statement in statements if isinstance(statement, dict)) if isinstance(statements, list) else ''
    numbers = normalized_numbers(statement_text)
    if numbers is None or not numbers.issubset(numeric_whitelist(card)):
        errors.append('В statement text есть числовое значение вне whitelist карточки.')
    if getattr(response, 'status', None) != 'completed':
        errors.append(f"API response status не completed: {getattr(response, 'status', None)}")
    if response_has_tool_call(response):
        errors.append('В response обнаружен tool call.')
    if not expected['synthetic'] or expected['execution_context']['real_client_data_used']:
        errors.append('Нарушен synthetic / no-real-client-data contract.')
    return {'ok': not errors, 'errors': errors}

# Минимальные детерминированные regression tests без API-вызовов.
case_a = FROZEN_CARDS['CASE_A']
assert numbers_are_allowed(case_a, 'threshold 0.50')
assert numbers_are_allowed(case_a, 'threshold 0,50')
assert numbers_are_allowed(case_a, 'threshold 0.5')
assert not numbers_are_allowed(case_a, 'threshold 0.75')
negative_provenance_response = {
    'scenario_id': 'CASE_A', 'recipient_role': 'sales_manager',
    'preserved_result': {key: case_a['model_result'][key] for key in ('model_name', 'score', 'threshold', 'relative_to_threshold', 'ml_decision', 'additional_review')},
    'statements': [
        {'kind': 'fact', 'text': 'Синтетическая карточка.', 'source_fact_ids': []},
        {'kind': 'fact', 'text': 'Score передан во входе.', 'source_fact_ids': ['model_result.score']}
    ],
    'used_factor_ids': []
}
offline_response = type('OfflineResponse', (), {'status': 'completed', 'output': []})()
negative_provenance_check = validate_response(case_a, 'sales_manager', offline_response, negative_provenance_response)
assert not negative_provenance_check['ok']
assert any('statement[0]' in error for error in negative_provenance_check['errors'])
factor_item_schema = INPUT_SCHEMA['properties']['local_factors']['items']
limitation_item_schema = INPUT_SCHEMA['properties']['limitations']['items']
assert set(factor_item_schema['required']) == {'factor_id', 'label', 'direction', 'provided_explanation'}
assert factor_item_schema['properties']['direction']['enum'] == ['raises_score', 'lowers_score', 'neutral']
assert set(limitation_item_schema['required']) == {'limitation_id', 'text'}
assert OUTPUT_SCHEMA['properties']['statements']['items']['properties']['source_fact_ids']['minItems'] == 1

def input_consistency_check():
    result = {}
    for scenario_id, card in FROZEN_CARDS.items():
        hashes = [sha256_text(canonical_json(card_for_role(card, role)['model_result'])) for role in ROLES]
        non_role_hashes = []
        for role in ROLES:
            payload = card_for_role(card, role)
            del payload['recipient_role']
            non_role_hashes.append(sha256_text(canonical_json(payload)))
        result[scenario_id] = {'model_result_canonical': len(set(hashes)) == 1, 'non_role_card_canonical': len(set(non_role_hashes)) == 1}
    return result

assert all(all(check.values()) for check in input_consistency_check().values())


## ФАКТЫ

Проверки отделяют подтверждение структуры и происхождения от проверки фактической истинности текста.

## ИНТЕРПРЕТАЦИЯ

Автоматический schema/provenance check не является доказательством, что все фактические claims в prose истинны; для этого предусмотрена отдельная ручная фиксация.

## ОГРАНИЧЕНИЯ

Регулярная проверка чисел детерминированна, но не заменяет содержательный review.

## СЛЕДУЮЩИЙ ШАГ

При наличии `OPENAI_API_KEY` выполнить только восемь предусмотренных API-вызовов.

In [20]:
# 4.1 — Ровно восемь независимых OpenAI Responses API calls

def usage_as_dict(usage):
    if usage is None:
        return None
    return usage.model_dump() if hasattr(usage, 'model_dump') else dict(usage)


def load_openai_api_key():
    """
    Загружает OPENAI_API_KEY.

    Приоритет:
    1. .env в корне проекта;
    2. переменная окружения текущего процесса.

    Сам ключ не выводится и не сохраняется в notebook outputs.
    """
    env_path = ROOT / '.env'

    if env_path.is_file():
        for raw_line in env_path.read_text(encoding='utf-8-sig').splitlines():
            line = raw_line.strip()

            if not line or line.startswith('#') or '=' not in line:
                continue

            name, value = line.split('=', 1)

            if name.strip() == 'OPENAI_API_KEY':
                api_key = value.strip().strip('"').strip("'")

                if api_key:
                    print(f'OPENAI_API_KEY загружен из {env_path.name}.')
                    return api_key

    api_key = os.environ.get('OPENAI_API_KEY')

    if api_key:
        print('OPENAI_API_KEY получен из переменной окружения.')
        return api_key

    print(
        'OPENAI_API_KEY не найден ни в .env проекта, '
        'ни в переменной окружения. API-раздел остановлен.'
    )
    return None


def run_api_calls():
    api_key = load_openai_api_key()

    if not api_key:
        print('Фиктивные результаты не создаются.')
        return []

    client = OpenAI(api_key=api_key)

    records = []
    started = time.perf_counter()

    role_labels = {
        'sales_manager': 'менеджер по продажам',
        'credit_controller': 'кредитный контролёр',
        'lawyer': 'юрист',
        'information_security': 'информационная безопасность',
    }

    planned_requests = [
        (scenario_id, role)
        for scenario_id in ('CASE_A', 'CASE_B')
        for role in ROLES
    ]

    assert len(planned_requests) == 8

    for index, (scenario_id, role) in enumerate(planned_requests, start=1):
        card = FROZEN_CARDS[scenario_id]
        role_label = role_labels[role]

        print(
            f'Запрос {index}/8 — {scenario_id} — {role_label} — '
            f'прошло {time.perf_counter() - started:.1f} с'
        )

        input_json = canonical_inputs[(scenario_id, role)]
        requested_at = utc_now()

        response = client.responses.create(
            model=MODEL_ID,
            instructions=MASTER_INSTRUCTION,
            input=input_json,
            reasoning={'effort': 'none'},
            store=False,
            tools=[],
            max_output_tokens=800,
            text={
                'format': {
                    'type': 'json_schema',
                    'name': 'stage20_role_interpretation',
                    'strict': True,
                    'schema': OUTPUT_SCHEMA,
                }
            },
        )

        try:
            parsed = json.loads(response.output_text)
            parse_error = None
        except (TypeError, json.JSONDecodeError) as error:
            parsed = None
            parse_error = str(error)

        if parsed is not None:
            validation = validate_response(
                card,
                role,
                response,
                parsed,
            )
        else:
            validation = {
                'ok': False,
                'errors': [
                    f'Structured Output parse неуспешен: {parse_error}'
                ],
            }

        records.append(
            {
                'scenario_id': scenario_id,
                'recipient_role': role,
                'response': parsed,
                'validation': validation,
                'metadata': {
                    'requested_model_id': MODEL_ID,
                    'response_model': getattr(response, 'model', None),
                    'utc_timestamp': requested_at,
                    'sdk_version': openai.__version__,
                    'response_id': getattr(response, 'id', None),
                    'response_status': getattr(response, 'status', None),
                    'usage': usage_as_dict(
                        getattr(response, 'usage', None)
                    ),
                    'master_prompt_sha256': MASTER_PROMPT_SHA256,
                    'canonical_input_sha256': sha256_text(input_json),
                },
            }
        )

    print(
        f'API-вызовы завершены: {len(records)}/8 — '
        f'общее время {time.perf_counter() - started:.1f} с'
    )

    return records


records = run_api_calls()

OPENAI_API_KEY загружен из .env.
Запрос 1/8 — CASE_A — менеджер по продажам — прошло 0.0 с
Запрос 2/8 — CASE_A — кредитный контролёр — прошло 4.1 с
Запрос 3/8 — CASE_A — юрист — прошло 7.7 с
Запрос 4/8 — CASE_A — информационная безопасность — прошло 13.4 с
Запрос 5/8 — CASE_B — менеджер по продажам — прошло 16.4 с
Запрос 6/8 — CASE_B — кредитный контролёр — прошло 20.0 с
Запрос 7/8 — CASE_B — юрист — прошло 23.3 с
Запрос 8/8 — CASE_B — информационная безопасность — прошло 27.3 с
API-вызовы завершены: 8/8 — общее время 29.9 с


In [21]:
# 4.2 — Сводка автоматической проверки и компактная таблица для ручного review
def review_rows(records):
    rows = []
    for record in records:
        rows.append({
            'сценарий': record['scenario_id'], 'роль': record['recipient_role'],
            'структура_OK': record['validation']['ok'],
            'факты_сохранены': record['validation']['ok'],
            'новые_числа_отсутствуют': record['validation']['ok'],
            'provenance_OK': record['validation']['ok'],
            'ручная_проверка_invented_factual_claims': 'ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ',
            'role_adaptation': 'ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ'
        })
    return rows

REVIEW_ROWS = review_rows(records)
for row in REVIEW_ROWS:
    print(' | '.join(map(str, row.values())))
passed = sum(record['validation']['ok'] for record in records)
print(f'Автоматически пройдено: {passed}/8')


CASE_A | sales_manager | True | True | True | True | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ
CASE_A | credit_controller | True | True | True | True | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ
CASE_A | lawyer | True | True | True | True | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ
CASE_A | information_security | True | True | True | True | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ
CASE_B | sales_manager | True | True | True | True | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ
CASE_B | credit_controller | True | True | True | True | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ
CASE_B | lawyer | True | True | True | True | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ
CASE_B | information_security | True | True | True | True | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ | ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ
Автоматически пройдено: 8/8


In [22]:
# 4.3 — Просмотр полученных интерпретаций и причин автоматической проверки

ROLE_LABELS_RU = {
    'sales_manager': 'Менеджер по продажам',
    'credit_controller': 'Кредитный контролёр',
    'lawyer': 'Юрист',
    'information_security': 'Информационная безопасность',
}

if not records:
    print('Нет полученных API-ответов.')
else:
    for index, record in enumerate(records, start=1):
        scenario_id = record['scenario_id']
        role = record['recipient_role']
        response = record.get('response')
        validation = record.get('validation', {})

        print('=' * 90)
        print(
            f'{index}/8 — {scenario_id} — '
            f'{ROLE_LABELS_RU.get(role, role)}'
        )
        print('-' * 90)

        if response is None:
            print('Ответ GPT не удалось разобрать.')
        else:
            preserved = response.get('preserved_result', {})

            print('Сохранённый результат модели:')
            print(
                f"  score = {preserved.get('score')} | "
                f"threshold = {preserved.get('threshold')} | "
                f"решение = {preserved.get('ml_decision')} | "
                f"доп. проверка = {preserved.get('additional_review')}"
            )

            print('\nИнтерпретация GPT:')

            statements = response.get('statements', [])

            if not statements:
                print('  Нет statements.')
            else:
                for statement_number, statement in enumerate(
                    statements,
                    start=1,
                ):
                    print(
                        f"  {statement_number}. "
                        f"{statement.get('text', '')}"
                    )
                    print(
                        '     Источники: '
                        + ', '.join(
                            statement.get('source_fact_ids', [])
                        )
                    )

            used_factor_ids = response.get('used_factor_ids', [])

            if used_factor_ids:
                print(
                    '\nИспользованные факторы: '
                    + ', '.join(used_factor_ids)
                )

        print('\nАвтоматическая проверка:')

        if validation.get('ok'):
            print('  PASS')
        else:
            print('  FAIL')

            errors = validation.get('errors', [])

            if errors:
                for error in errors:
                    print(f'  - {error}')
            else:
                print('  - Причина не указана.')

        print()

1/8 — CASE_A — Менеджер по продажам
------------------------------------------------------------------------------------------
Сохранённый результат модели:
  score = 0.32 | threshold = 0.5 | решение = standard_processing | доп. проверка = False

Интерпретация GPT:
  1. Синтетическая модель SYNTHETIC_GBDT_DEMO вернула score 0.32, что ниже threshold 0.5. Зафиксированное решение модели — standard_processing; дополнительная проверка не требуется.
     Источники: model_result.model_name, model_result.score, model_result.threshold, model_result.relative_to_threshold, model_result.ml_decision, model_result.additional_review
  2. Для указанного workflow результат означает стандартную обработку без дополнительной проверки.
     Источники: model_result.ml_decision, model_result.additional_review, model_result.decision_mode
  3. Карточка синтетическая и не содержит реальных клиентских данных; интерпретация не является кредитным решением и не включает расчёт кредитного риска.
     Источники: limi

## ФАКТЫ

Таблица создаётся только из реальных API responses. В ней намеренно не проставляется автоматический PASS для invented factual claims и role adaptation.

## ИНТЕРПРЕТАЦИЯ

Статус `READY` допустим только при 8/8 успешных автоматических проверках и отдельной зафиксированной ручной проверке всех восьми outputs.

## ОГРАНИЧЕНИЯ

До выполнения API calls либо до завершения ручной проверки accepted artifact не создаётся.

## СЛЕДУЮЩИЙ ШАГ

Вручную заполнить `MANUAL_REVIEW_RECORD` после чтения восьми outputs, затем сохранить artifact.

In [5]:
# 4.4 — Восстановление уже полученных 8 API-ответов из сохранённого artifact

saved_artifact = json.loads(
    ARTIFACT_PATH.read_text(encoding='utf-8')
)

assert saved_artifact['api_calls_executed'] == 8
assert saved_artifact['automated_validation']['passed'] == 8
assert saved_artifact['automated_validation']['required'] == 8
assert saved_artifact['automated_validation']['status'] == 'PASSED'

records = saved_artifact['responses']

assert len(records) == 8
assert all(
    record['validation']['ok']
    for record in records
)

print(
    f'Восстановлено сохранённых API-ответов: {len(records)}/8. '
    'Новые API-вызовы не выполнялись.'
)

Восстановлено сохранённых API-ответов: 8/8. Новые API-вызовы не выполнялись.


In [6]:
# 5.1 — Явная ручная фиксация и сохранение accepted artifact

MANUAL_CONFIRMATION = 'ПОДТВЕРЖДЕНО_РУЧНОЙ_ПРОВЕРКОЙ'

MANUAL_REVIEW_RECORD = [
    {
        'scenario_id': 'CASE_A',
        'recipient_role': 'sales_manager',
        'invented_factual_claims': MANUAL_CONFIRMATION,
        'role_adaptation': MANUAL_CONFIRMATION,
    },
    {
        'scenario_id': 'CASE_A',
        'recipient_role': 'credit_controller',
        'invented_factual_claims': MANUAL_CONFIRMATION,
        'role_adaptation': MANUAL_CONFIRMATION,
    },
    {
        'scenario_id': 'CASE_A',
        'recipient_role': 'lawyer',
        'invented_factual_claims': MANUAL_CONFIRMATION,
        'role_adaptation': MANUAL_CONFIRMATION,
    },
    {
        'scenario_id': 'CASE_A',
        'recipient_role': 'information_security',
        'invented_factual_claims': MANUAL_CONFIRMATION,
        'role_adaptation': MANUAL_CONFIRMATION,
    },
    {
        'scenario_id': 'CASE_B',
        'recipient_role': 'sales_manager',
        'invented_factual_claims': MANUAL_CONFIRMATION,
        'role_adaptation': MANUAL_CONFIRMATION,
    },
    {
        'scenario_id': 'CASE_B',
        'recipient_role': 'credit_controller',
        'invented_factual_claims': MANUAL_CONFIRMATION,
        'role_adaptation': MANUAL_CONFIRMATION,
    },
    {
        'scenario_id': 'CASE_B',
        'recipient_role': 'lawyer',
        'invented_factual_claims': MANUAL_CONFIRMATION,
        'role_adaptation': MANUAL_CONFIRMATION,
    },
    {
        'scenario_id': 'CASE_B',
        'recipient_role': 'information_security',
        'invented_factual_claims': MANUAL_CONFIRMATION,
        'role_adaptation': MANUAL_CONFIRMATION,
    },
]


def manual_review_complete(manual_record):
    expected = {
        (scenario_id, role)
        for scenario_id in FROZEN_CARDS
        for role in ROLES
    }

    if not isinstance(manual_record, list) or len(manual_record) != 8:
        return False

    actual = {
        (item.get('scenario_id'), item.get('recipient_role'))
        for item in manual_record
    }

    return (
        actual == expected
        and all(
            item.get('invented_factual_claims') == MANUAL_CONFIRMATION
            and item.get('role_adaptation') == MANUAL_CONFIRMATION
            for item in manual_record
        )
    )


def build_review_rows(records, manual_record):
    manual_by_key = {
        (item['scenario_id'], item['recipient_role']): item
        for item in manual_record
    } if isinstance(manual_record, list) else {}

    rows = []

    for record in records:
        key = (
            record['scenario_id'],
            record['recipient_role'],
        )

        manual_item = manual_by_key.get(key, {})

        rows.append({
            'сценарий': record['scenario_id'],
            'роль': record['recipient_role'],
            'структура_OK': record['validation']['ok'],
            'факты_сохранены': record['validation']['ok'],
            'новые_числа_отсутствуют': record['validation']['ok'],
            'provenance_OK': record['validation']['ok'],
            'ручная_проверка_invented_factual_claims':
                manual_item.get(
                    'invented_factual_claims',
                    'ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ',
                ),
            'role_adaptation':
                manual_item.get(
                    'role_adaptation',
                    'ОЖИДАЕТ_РУЧНОЙ_ПРОВЕРКИ',
                ),
        })

    return rows


def build_artifact(records, manual_record):
    passed = sum(
        record['validation']['ok']
        for record in records
    )

    automation_ok = (
        len(records) == 8
        and passed == 8
    )

    manual_ok = manual_review_complete(
        manual_record
    )

    status = (
        'ROLE_BASED_RESULT_INTERPRETER_PROTOTYPE_READY'
        if automation_ok and manual_ok
        else 'AWAITING_API_EXECUTION_AND_OR_MANUAL_REVIEW'
    )

    validation_status = (
        'NOT_RUN'
        if not records
        else (
            'PASSED'
            if automation_ok
            else 'INCOMPLETE_OR_FAILED'
        )
    )

    return {
        'stage': 'Stage 20 V1',
        'status': status,
        'accepted': automation_ok and manual_ok,
        'contract': {
            'requested_model_id': MODEL_ID,
            'reasoning_effort': 'none',
            'store': False,
            'tools': [],
            'max_output_tokens': 800,
            'real_client_data_used': False,
            'frozen_synthetic_cards': [
                'CASE_A',
                'CASE_B',
            ],
            'roles': list(ROLES),
        },
        'master_prompt_sha256': MASTER_PROMPT_SHA256,
        'api_calls_executed': len(records),
        'automated_validation': {
            'passed': passed,
            'required': 8,
            'status': validation_status,
            'input_card_invariants_by_scenario':
                input_consistency_check(),
        },
        'manual_review': {
            'required': True,
            'status': (
                'COMPLETED'
                if manual_ok
                else 'REQUIRED'
            ),
            'record': manual_record,
        },
        'review_table': build_review_rows(
            records,
            manual_record,
        ),
        'responses': records,
    }


def write_artifact(records, manual_record):
    artifact = build_artifact(
        records,
        manual_record,
    )

    ARTIFACT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    ARTIFACT_PATH.write_text(
        json.dumps(
            artifact,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        ) + '\n',
        encoding='utf-8',
    )

    return artifact


assert len(records) == 8
assert all(
    record['validation']['ok']
    for record in records
)
assert manual_review_complete(
    MANUAL_REVIEW_RECORD
)

artifact = write_artifact(
    records,
    MANUAL_REVIEW_RECORD,
)

assert artifact['accepted'] is True
assert artifact['status'] == (
    'ROLE_BASED_RESULT_INTERPRETER_PROTOTYPE_READY'
)
assert artifact['manual_review']['status'] == 'COMPLETED'

print(f'Accepted artifact сохранён: {ARTIFACT_PATH}')
print(
    'Stage 20: '
    f"{artifact['automated_validation']['passed']}/8 automated PASS; "
    '8/8 manual PASS; '
    f"status = {artifact['status']}"
)

Accepted artifact сохранён: D:\Projects\komus-work\reports\generated\stage20_role_interpreter_V1.json
Stage 20: 8/8 automated PASS; 8/8 manual PASS; status = ROLE_BASED_RESULT_INTERPRETER_PROTOTYPE_READY


In [7]:
# 6.1 — Читаемый итог: как один результат объясняется разным ролям

import json
from pathlib import Path
from IPython.display import display, Markdown


def find_project_root_for_report(start_path):
    for candidate in (
        start_path.resolve(),
        *start_path.resolve().parents,
    ):
        if (
            (candidate / 'pyproject.toml').is_file()
            and (candidate / 'reports').is_dir()
            and (candidate / 'notebooks').is_dir()
        ):
            return candidate

    raise RuntimeError(
        'Не удалось определить корень проекта.'
    )


root = find_project_root_for_report(Path.cwd())

artifact_path = (
    root
    / 'reports'
    / 'generated'
    / 'stage20_role_interpreter_V1.json'
)

artifact = json.loads(
    artifact_path.read_text(encoding='utf-8')
)


ROLE_LABELS = {
    'sales_manager': 'Менеджер по продажам',
    'credit_controller': 'Кредитный контролёр',
    'lawyer': 'Юрист',
    'information_security': 'Информационная безопасность',
}


SCENARIO_LABELS = {
    'CASE_A': 'Обычная ситуация: стандартная обработка',
    'CASE_B': 'Пограничная ситуация: дополнительная проверка',
}


DECISION_LABELS = {
    'standard_processing': 'стандартная обработка',
    'additional_review': 'дополнительная проверка',
}


RELATIVE_LABELS = {
    'below': 'ниже порога',
    'equal': 'равен порогу',
    'above': 'выше порога',
}


responses = artifact.get('responses', [])

if len(responses) != 8:
    raise RuntimeError(
        f'Ожидалось 8 сохранённых ответов, найдено: {len(responses)}'
    )


auto = artifact.get('automated_validation', {})
manual = artifact.get('manual_review', {})

summary = [
    '# Stage 20 — Ролевой интерпретатор результата',
    '',
    '## Итог эксперимента',
    '',
    f"- Сохранённых ответов GPT: **{len(responses)}/8**",
    (
        f"- Автоматическая проверка: "
        f"**{auto.get('passed', 0)}/{auto.get('required', 8)}**"
    ),
    (
        f"- Ручная проверка: "
        f"**{manual.get('status', 'НЕ ЗАВЕРШЕНА')}**"
    ),
    '',
    (
        'Во всех случаях исходный ML-результат уже рассчитан заранее. '
        'GPT его не пересчитывает и не меняет — меняется только форма объяснения '
        'для конкретной рабочей роли.'
    ),
]

display(Markdown('\n'.join(summary)))


for scenario_id in ('CASE_A', 'CASE_B'):

    scenario_records = [
        record
        for record in responses
        if record['scenario_id'] == scenario_id
    ]

    first_response = scenario_records[0]['response']
    result = first_response['preserved_result']

    scenario_header = [
        '',
        '---',
        '',
        f"## {scenario_id} — {SCENARIO_LABELS[scenario_id]}",
        '',
        '### Зафиксированный результат модели',
        '',
        f"- **Score:** `{result['score']}`",
        f"- **Порог:** `{result['threshold']}`",
        (
            f"- **Положение относительно порога:** "
            f"{RELATIVE_LABELS.get(result['relative_to_threshold'], result['relative_to_threshold'])}"
        ),
        (
            f"- **Режим:** "
            f"{DECISION_LABELS.get(result['ml_decision'], result['ml_decision'])}"
        ),
        (
            f"- **Дополнительная проверка:** "
            f"{'да' if result['additional_review'] else 'нет'}"
        ),
        '',
        '### Как этот же результат объясняется разным ролям',
    ]

    display(Markdown('\n'.join(scenario_header)))

    for role in (
        'sales_manager',
        'credit_controller',
        'lawyer',
        'information_security',
    ):

        record = next(
            item
            for item in scenario_records
            if item['recipient_role'] == role
        )

        response = record['response']
        validation = record['validation']

        role_block = [
            f"#### {ROLE_LABELS[role]}",
            '',
        ]

        for statement in response.get('statements', []):
            role_block.append(
                f"- {statement['text']}"
            )

        role_block.extend([
            '',
            (
                f"**Контроль ответа:** "
                f"{'PASS' if validation.get('ok') else 'FAIL'}"
            ),
            '',
        ])

        display(Markdown('\n'.join(role_block)))

# Stage 20 — Ролевой интерпретатор результата

## Итог эксперимента

- Сохранённых ответов GPT: **8/8**
- Автоматическая проверка: **8/8**
- Ручная проверка: **COMPLETED**

Во всех случаях исходный ML-результат уже рассчитан заранее. GPT его не пересчитывает и не меняет — меняется только форма объяснения для конкретной рабочей роли.


---

## CASE_A — Обычная ситуация: стандартная обработка

### Зафиксированный результат модели

- **Score:** `0.32`
- **Порог:** `0.5`
- **Положение относительно порога:** ниже порога
- **Режим:** стандартная обработка
- **Дополнительная проверка:** нет

### Как этот же результат объясняется разным ролям

#### Менеджер по продажам

- Синтетическая модель SYNTHETIC_GBDT_DEMO вернула score 0.32, что ниже threshold 0.5. Зафиксированное решение модели — standard_processing; дополнительная проверка не требуется.
- Для указанного workflow результат означает стандартную обработку без дополнительной проверки.
- Карточка синтетическая и не содержит реальных клиентских данных; интерпретация не является кредитным решением и не включает расчёт кредитного риска.

**Контроль ответа:** PASS


#### Кредитный контролёр

- Модель SYNTHETIC_GBDT_DEMO присвоила score 0.32 при threshold 0.5; score находится ниже threshold.
- Зафиксированное решение модели: standard_processing. Дополнительная проверка не требуется по переданному результату.
- Переданные локальные факторы: стабильная платёжная дисциплина и финансовая устойчивость без отмеченных отклонений.
- Карточка синтетическая и не содержит реальных клиентских данных; интерпретация не является кредитным решением, а GPT не рассчитывает кредитный риск.

**Контроль ответа:** PASS


#### Юрист

- Результат получен от синтетической модели SYNTHETIC_GBDT_DEMO в зафиксированном workflow-режиме из frozen card.
- Модельный score равен 0.32; он ниже threshold 0.5. ML-решение указано как standard_processing, дополнительная проверка не требуется согласно карточке.
- В карточке указаны локальные факторы: стабильная платёжная дисциплина и финансовая устойчивость без отмеченных отклонений.
- Карточка синтетическая и не содержит реальных клиентских данных.
- Интерпретация не является кредитным решением и не включает расчёт кредитного риска.
- Внешние инструменты не предоставлены.

**Контроль ответа:** PASS


...


---

## CASE_B — Пограничная ситуация: дополнительная проверка

### Зафиксированный результат модели

- **Score:** `0.49`
- **Порог:** `0.5`
- **Положение относительно порога:** ниже порога
- **Режим:** дополнительная проверка
- **Дополнительная проверка:** да

### Как этот же результат объясняется разным ролям

#### Менеджер по продажам

- Модель SYNTHETIC_GBDT_DEMO выдала score 0.49 при threshold 0.5; score находится ниже threshold.
- Зафиксированный результат workflow — additional_review: требуется дополнительное рассмотрение.
- В карточке локально переданы факторы: платёжная дисциплина и финансовая устойчивость; оба требуют дополнительного внимания.
- Карточка синтетическая и не содержит реальных клиентских данных; интерпретация не является кредитным решением.

**Контроль ответа:** PASS


#### Кредитный контролёр

- Модель SYNTHETIC_GBDT_DEMO присвоила score 0.49 при threshold 0.5; score находится ниже threshold.
- Зафиксированное решение модели — additional_review; дополнительная проверка указана как требуемая.
- Переданные локальные факторы: платёжная дисциплина и финансовая устойчивость требуют дополнительного внимания.
- Карточка синтетическая и не содержит реальных клиентских данных. Интерпретация не является кредитным решением; GPT не рассчитывает кредитный риск. Внешние инструменты не предоставлены.

**Контроль ответа:** PASS


#### Юрист

- Результат получен моделью SYNTHETIC_GBDT_DEMO; score указан как синтетический модельный score из frozen card и находится ниже threshold.
- В карточке зафиксирован ml_decision additional_review и признак additional_review=true.
- В качестве локально переданных факторов указаны платёжная дисциплина и финансовая устойчивость; оба фактора требуют дополнительного внимания по переданным пояснениям.
- Карточка синтетическая и не содержит реальных клиентских данных. Интерпретация не является кредитным решением и не представляет собой расчёт кредитного риска.
- Внешние инструменты не предоставлены.

**Контроль ответа:** PASS


#### Информационная безопасность

- Карточка синтетическая и не содержит реальных клиентских данных.
- В текущем execution_context внешние инструменты, веб-поиск и file search не предоставлены.
- Параметр store передан со значением false.

**Контроль ответа:** PASS


## ФАКТЫ

Stage 20 завершён успешно.

- Выполнено **8 реальных OpenAI Responses API-вызовов**: 2 синтетические карточки × 4 бизнес-роли.
- Автоматическая проверка пройдена: **8/8 PASS**.
- Ручная проверка пройдена: **8/8 PASS**.
- Финальный статус эксперимента: `ROLE_BASED_RESULT_INTERPRETER_PROTOTYPE_READY`.
- Во всех восьми случаях сохранены исходные `score`, `threshold`, `ml_decision` и `additional_review`.
- GPT не выполнял новый расчёт кредитного риска и не менял результат ML-модели.
- Для одного и того же результата форма и акценты объяснения менялись в зависимости от роли:
  - менеджер по продажам;
  - кредитный контролёр;
  - юрист;
  - информационная безопасность.
- API key загружается локально из `.env` проекта с резервным чтением `OPENAI_API_KEY` из переменной окружения. Значение ключа не выводится в notebook и не записывается в итоговый JSON artifact.
- Для каждого реального вызова сохранены технические данные воспроизводимости: requested model ID, response model, UTC timestamp, SDK version, response ID, status, usage и SHA-256 контрольных представлений prompt/input.
- Финальный evidence artifact сохранён в `reports/generated/stage20_role_interpreter_V1.json`.

## ИНТЕРПРЕТАЦИЯ

Эксперимент подтверждает, что при неизменном заранее рассчитанном ML-результате LLM можно использовать как **Result Interpreter**: один и тот же набор фактов воспроизводимо представляется с разными акцентами для разных рабочих ролей.

При этом граница между ML и LLM сохраняется:

- ML-модель рассчитывает риск и формирует исходный результат;
- GPT только объясняет уже переданный результат;
- GPT не является кредитным предиктором и не принимает кредитное решение.

Синтетические карточки использованы намеренно, чтобы изолировать исследовательский вопрос и не передавать в внешний API реальные клиентские данные.

## ОГРАНИЧЕНИЯ

Stage 20 не доказывает:

- улучшение `Gini`, `Recall`, `PR-AUC` или других метрик кредитной модели;
- улучшение качества кредитного решения;
- юридическую достаточность формулировок;
- production/security readiness;
- полезность интерфейса для реальных пользователей;
- детерминированность текста между повторными LLM-вызовами;
- способность GPT самостоятельно строить объяснение модели без заранее подготовленного набора фактов.

`store=false` является параметром API-запроса и не трактуется как Zero Data Retention или как гарантия полного отсутствия хранения данных.

## СЛЕДУЮЩИЙ ШАГ

Stage 20 больше не требует новых API-вызовов.

Следующий шаг:

1. проверить финальный diff и отсутствие секретов;
2. сохранить notebook и accepted JSON artifact в Git;
3. обновить состояние проекта;
4. синхронизировать принятую версию;
5. перейти к сборке итоговой презентации и evidence package для защиты.